In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("sample_RDD").getOrCreate()
number =[10,20,30,40,50]

sc = spark.sparkContext

rdd = sc.parallelize(number)
print(type(rdd))
print(rdd.collect())

<class 'pyspark.core.rdd.RDD'>
[10, 20, 30, 40, 50]


In [ ]:
#  mulitply each value in rdd with 5

rdd2= rdd.map(lambda x: x*5)
print(rdd2.collect())


rdd2= rdd.map(lambda i: i +5)
print(rdd2.collect())

# create new RDD with values >= 40
rdd3 = rdd.filter( lambda x : x >= 40)
print(rdd3.collect())

[50, 100, 150, 200, 250]
[15, 25, 35, 45, 55]
[40, 50]


In [ ]:
strings = ['hello', 'students', 'pyspark']

str_rdd= sc.parallelize(strings)
upp_rdd = str_rdd.map(lambda x : len(x))
print(upp_rdd.collect())
print(upp_rdd.count())

[5, 8, 7]
3


In [ ]:
l1 =['hi students, who are you? pyspark session started']

rdd1= sc.parallelize(l1)
rdd2= rdd1.flatMap(lambda x: x.split(' '))
print(rdd2.collect())
print(rdd2.count())

['hi', 'students,', 'who', 'are', 'you?', 'pyspark', 'session', 'started']
8


In [ ]:
s1='hi students, who are you? pyspark session started'
print(s1.split(' '))

['hi', 'students,', 'who', 'are', 'you?', 'pyspark', 'session', 'started']


In [ ]:
l1 =[['hi students,'], ['who are you?'], ['pyspark session'], ['started']]

rdd1= sc.parallelize(l1)
rdd2= rdd1.flatMap(lambda x: x[0].split(' '))
print(rdd2.collect())
print(rdd2.count())
print(rdd2.take(6))
print(rdd2.first())

['hi', 'students,', 'who', 'are', 'you?', 'pyspark', 'session', 'started']
8
['hi', 'students,', 'who', 'are', 'you?', 'pyspark']
hi


In [ ]:
l1 =[['hi students, hi'], ['hi hello you?'], ['pyspark session'], ['pyspark']]

rdd1= sc.parallelize(l1)
# print how many times each word is presented on RDD
rdd2= rdd1.flatMap(lambda x: x[0].split(' '))


map_rdd = rdd2.map( lambda x : (x,1))
red_rdd = map_rdd.reduceByKey( lambda a,b : a+b)
print(red_rdd.collect())

grp_rdd = rdd2.groupBy(lambda word:word)
res_rdd = grp_rdd.mapValues(len)
print(res_rdd.collect())


[('students,', 1), ('hello', 1), ('session', 1), ('hi', 3), ('you?', 1), ('pyspark', 2)]
[('students,', 1), ('hello', 1), ('session', 1), ('hi', 3), ('you?', 1), ('pyspark', 2)]


In [ ]:
l=[10,20,30,405,60,708,8]

num_rdd = sc.parallelize(l)
print(num_rdd.collect())

res_rdd = num_rdd.reduce(lambda a,b : a+b)
print(res_rdd)


[10, 20, 30, 405, 60, 708, 8]
13.985185185185184


In [ ]:
rdd = sc.textFile('orders.csv')
print(rdd.collect())

# find the no of completed orders for each category
header = rdd.first()
data_rdd = rdd.filter(lambda line : line !=header)

orders_rdd = data_rdd.map(lambda line : line.split(','))

print(orders_rdd.collect())

complete_rdd = orders_rdd.filter(lambda row: row[5] == 'COMPLETED')
print(complete_rdd.collect())

# count by category
category_count_rdd = complete_rdd.map(lambda row : (row[3], 1))
result = category_count_rdd.reduceByKey(lambda x,y : x+y)
print(result.collect())

result.coalesce(1).saveAsTextFile("results_new2")

['order_id,customer_id,product,category,amount,status', '1001,C101,Laptop,Electronics,75000,COMPLETED', '1002,C102,Mouse,Electronics,1500,COMPLETED', '1003,C101,Keyboard,Electronics,3000,CANCELLED', '1004,C103,Sofa,Furniture,45000,COMPLETED', '1005,C102,Chair,Furniture,8000,COMPLETED', '1006,C101,Monitor,Electronics,25000,COMPLETED', '1007,C104,Table,Furniture,12000,CANCELLED', '1008,C103,Laptop,Electronics,80000,COMPLETED', '1009,C102,Desk,Furniture,15000,COMPLETED', '1010,C101,Headphones,Electronics,5000,COMPLETED']
[['1001', 'C101', 'Laptop', 'Electronics', '75000', 'COMPLETED'], ['1002', 'C102', 'Mouse', 'Electronics', '1500', 'COMPLETED'], ['1003', 'C101', 'Keyboard', 'Electronics', '3000', 'CANCELLED'], ['1004', 'C103', 'Sofa', 'Furniture', '45000', 'COMPLETED'], ['1005', 'C102', 'Chair', 'Furniture', '8000', 'COMPLETED'], ['1006', 'C101', 'Monitor', 'Electronics', '25000', 'COMPLETED'], ['1007', 'C104', 'Table', 'Furniture', '12000', 'CANCELLED'], ['1008', 'C103', 'Laptop', 'Ele

In [ ]:

orders_df = spark.read.csv('orders.csv', header= True)

orders_df.show()

result_df = orders_df.filter(orders_df.status == 'COMPLETED') \
                      .groupBy("category") \
                      .count()
result_df.show()


+--------+-----------+----------+-----------+------+---------+
|order_id|customer_id|   product|   category|amount|   status|
+--------+-----------+----------+-----------+------+---------+
|    1001|       C101|    Laptop|Electronics| 75000|COMPLETED|
|    1002|       C102|     Mouse|Electronics|  1500|COMPLETED|
|    1003|       C101|  Keyboard|Electronics|  3000|CANCELLED|
|    1004|       C103|      Sofa|  Furniture| 45000|COMPLETED|
|    1005|       C102|     Chair|  Furniture|  8000|COMPLETED|
|    1006|       C101|   Monitor|Electronics| 25000|COMPLETED|
|    1007|       C104|     Table|  Furniture| 12000|CANCELLED|
|    1008|       C103|    Laptop|Electronics| 80000|COMPLETED|
|    1009|       C102|      Desk|  Furniture| 15000|COMPLETED|
|    1010|       C101|Headphones|Electronics|  5000|COMPLETED|
+--------+-----------+----------+-----------+------+---------+

+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|    5|
|  Furniture|    3|
+-----------+----

In [ ]:
columns =['id', 'name']
data =[(1, 'charan'), (2, 'ram'), (3, 'rahul'), (None, None), ('arun', 'vikaram')]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

+----+-------+
|  id|   name|
+----+-------+
|   1| charan|
|   2|    ram|
|   3|  rahul|
|NULL|   NULL|
|arun|vikaram|
+----+-------+

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)



In [ ]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, BooleanType, DoubleType, DateType

schema = StructType([StructField('id', IntegerType(), False),
                     StructField('name', StringType(), False)])

data =[(1, 'charan'), (2, 'ram'), (3, 'rahul'), (4, 'arun'), (5, 'vikaram')]

df = spark.createDataFrame(data, schema= schema)
df.show()

+---+-------+
| id|   name|
+---+-------+
|  1| charan|
|  2|    ram|
|  3|  rahul|
|  4|   arun|
|  5|vikaram|
+---+-------+



In [ ]:
# emp_df ==> id int, name varchar, status boolean, doj datetype, salary double
from datetime import date
emp_schema = StructType([ StructField('id', IntegerType()),
                          StructField('name', StringType()),
                          StructField('city', StringType()),
                          StructField('status', BooleanType()),
                          StructField('DOJ', DateType()),
                          StructField('salary', DoubleType())])

emp_data = [(1, 'bangalore', 'charan', True, date(2023,1,1), 5000.78)]

emp_df = spark.createDataFrame(data=emp_data, schema=emp_schema)

emp_df.show()

+---+---------+------+------+----------+-------+
| id|     name|  city|status|       DOJ| salary|
+---+---------+------+------+----------+-------+
|  1|bangalore|charan|  true|2023-01-01|5000.78|
+---+---------+------+------+----------+-------+



order_id,customer_id,product,category,amount,status
1001,C101,Laptop,Electronics,75000,COMPLETED
1002,C102,Mouse,Electronics,1500,COMPLETED
1003,C101,Keyboard,Electronics,3000,CANCELLED
1004,C103,Sofa,Furniture,45000,COMPLETED
1005,C102,Chair,Furniture,8000,COMPLETED
1006,C101,Monitor,Electronics,25000,COMPLETED
1007,C104,Table,Furniture,12000,CANCELLED
1008,C103,Laptop,Electronics,80000,COMPLETED
1009,C102,Desk,Furniture,15000,COMPLETED
1010,C101,Headphones,Electronics,5000,COMPLETED

spark works on laxy evolution
untill an action triggers no transformation will not execute

transformations:
---------------
filter
group by
join
partition

actions:-
---------
show
collect
count
write



=======================


job(for every action, 1 job will be created)  ==> stage  ==> task


transformations -->
1. narrow transformation  --> where the no shuffling of data
	where/filter
	select/drop

2. wider transformation   --> data shuffles which leads to multiple stages
	groupBy()
	orderBy()
	sortBy()
	join()
	distinct

file formats:-


1. text/csv  --> row based  ==> 1GB == XXXXXXXXXXXXXXXXXX

2. avro  --> row based      ==> 600MB(40% compression)==> data validation(key value pair)

3. parquet  --> columnar based  ==> ~400MB(55 - 65% compression)  ==> king of all  file formats ==> 400GB (40min) ==> 40rs(proc) + 15 (storage) ==> 55 rs  --> daily jobs
4. ORC --> columnar based    ==> ~100MB(90-95% compression)   ==> 100GB(60min) ==> 60rs(processing) + 10 rs(storage) ==> 70rs
	--> when jobs run very often(monthly, quarterly, yearly, weekly)



row based file format


column based file format